# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-04-14

### Surrogate model training script (REFACTORED)
**Goal:** Train a Graph Neural Network surrogate model on structural analysis data to predict beam forces.

**Architecture:** This notebook now uses a modular design:
- **`c21_train.py`**: Core training logic (data loading, processing, training, export)
- **`c21_surrogate_model_training.ipynb`**: Orchestration and interactive visualization

**Usage:**
1. Set environment variables or use defaults (see `c21_train.py`)
2. Run the training cell (Cell 2) → executes complete workflow
3. Run visualization cells (Cells 3-5) → generate diagnostic plots
4. Run evaluation cell (Cell 6) → export metrics

**Inputs:**
- CSV files with structural properties from Grasshopper (node, edge, global data)

**Outputs:**
- Trained surrogate model checkpoint
- Scalers for inference
- Evaluation metrics and diagnostic plots


# TRAINING EXECUTION

All training logic is encapsulated in src/c21_train.py. Run the cell below to execute the complete workflow (data loading -> processing -> training -> export). Parameters are set via environment variables (see src/c21_train.py or set via os.environ before running).


In [1]:
"""
RUN TRAINING WORKFLOW - Single cell execution
This notebook now uses src/c21_train.py for core logic.
"""
from src.c21_train import main

# Execute training
results = main()

# Extract results for visualization cells below
model = results["model"]
device = results["device"]
epoch_history = results["epoch_history"]
train_loss_history = results["train_loss_history"]
final_val_r2 = results["final_val_r2"]
edge_target_scaler = results["scalers"]["edge_target"]
train_loader = results["train_loader"]
test_loader = results["test_loader"]
schema = results["schema"]
params = results["params"]

print("Training workflow complete. Results available for visualization.")


c:\Users\Jasper\Documents\PyEnvs\thesis_home_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


System loaded successfully.

Code is running locally from: thesis_generative_timber
Data connected to OneDrive: 2.2 - 2.4

GH data directory: C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\01_grasshopper_data
Raw data directory: C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\30_Data_Inventory\02_raw_data
Export directory: C:\Users\Jasper\OneDrive\06 Building Technology TU\2.2 - 2.4\60_Research_Exports
c21 SURROGATE MODEL TRAINING

Device: cpu

Parameters loaded:
- lr=0.0005, epochs=100, batch=16
- hidden_dim=128, weight_decay=0.0
- Run ID: ID20260414_194854

1. Loading multi-source dataset...

--- DATA VALIDATION ---
Node rows:         130000
Edge rows:         320000
Global rows:       10000
Samples:           10000
Node count:        13
Edge count:        32
edge_index shape:  (2, 32)
Validation successful. Multi-source data loaded correctly.

Processing and normalizing data...
Dataset ready! Train: 8000 graphs. Test: 2000 graphs.

Traini

KeyboardInterrupt: 

In [ ]:
# Plot: Loss curve + normalized target distribution on test set
import numpy as np
import torch
from model_evaluation import build_training_visuals_figure

model.eval()
all_test_targets_scaled = []
all_test_preds_scaled = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device, non_blocking=True)
        out = model(
            batch.x,
            batch.edge_index,
            edge_attr=batch.edge_attr,
            batch=batch.batch,
            u=batch.u,
        )
        all_test_targets_scaled.append(batch.y_edge.detach().cpu().numpy())
        all_test_preds_scaled.append(out.detach().cpu().numpy())

test_targets_scaled = np.concatenate(all_test_targets_scaled, axis=0) if len(all_test_targets_scaled) > 0 else np.array([])
test_preds_scaled = np.concatenate(all_test_preds_scaled, axis=0) if len(all_test_preds_scaled) > 0 else np.array([])

training_visuals_fig = build_training_visuals_figure(
    epoch_history,
    train_loss_history,
    test_targets_scaled,
    test_preds_scaled,
 )

training_visuals_fig

# EVALUATION & VISUALIZATION FOR OVER/UNDERFITTING

In [ ]:
"""Setup for visualization cells - prepare environment and imports"""
import matplotlib.pyplot as plt
import numpy as np
import torch
import config
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Environment ready for visualization.")


In [ ]:
# Predictions vs Actual + Residual diagnostics (Train/Test)
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from model_evaluation import build_pred_residual_figure

def _collect_preds_trues_original(loader):
    model.eval()
    pred_batches_local = []
    true_batches_local = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device, non_blocking=True)
            out = model(
                batch.x,
                batch.edge_index,
                edge_attr=batch.edge_attr,
                batch=batch.batch,
                u=batch.u,
            )
            pred_batches_local.append(out.detach().cpu().numpy())
            true_batches_local.append(batch.y_edge.detach().cpu().numpy())

    preds_scaled_local = np.concatenate(pred_batches_local, axis=0)
    trues_scaled_local = np.concatenate(true_batches_local, axis=0)

    preds_original_local = edge_target_scaler.inverse_transform(preds_scaled_local).reshape(-1)
    trues_original_local = edge_target_scaler.inverse_transform(trues_scaled_local).reshape(-1)
    return preds_original_local, trues_original_local

# Collect predictions in original kN scale
train_preds_original, train_trues_original = _collect_preds_trues_original(train_loader)
test_preds_original, test_trues_original = _collect_preds_trues_original(test_loader)

# Residuals and metrics
train_residuals = train_trues_original - train_preds_original
test_residuals = test_trues_original - test_preds_original

train_mae = float(mean_absolute_error(train_trues_original, train_preds_original))
test_mae = float(mean_absolute_error(test_trues_original, test_preds_original))
train_rmse = float(np.sqrt(mean_squared_error(train_trues_original, train_preds_original)))
test_rmse = float(np.sqrt(mean_squared_error(test_trues_original, test_preds_original)))
train_r2 = float(r2_score(train_trues_original, train_preds_original))
test_r2 = float(r2_score(test_trues_original, test_preds_original))

print(f"Train R2:  {train_r2:.4f}")
print(f"Test R2:   {test_r2:.4f}")
print(f"Train MAE: {train_mae:.4f} kN")
print(f"Test MAE:  {test_mae:.4f} kN")
print(f"Train RMSE: {train_rmse:.4f} kN")
print(f"Test RMSE:  {test_rmse:.4f} kN\n")

pred_residuals_fig = build_pred_residual_figure(
    train_trues_original,
    train_preds_original,
    test_trues_original,
    test_preds_original,
    train_r2,
    test_r2,
 )

pred_residuals_fig

In [ ]:
# Error distribution plots
from model_evaluation import build_error_distribution_figure

error_dist_fig = build_error_distribution_figure(
    train_residuals,
    test_residuals,
    train_mae,
    test_mae,
 )

error_dist_fig

In [ ]:
# Final evaluation export
import importlib
from model_evaluation import save_evaluation
from naming import build_model_artifact_stem

# Determine fit status
r2_gap = train_r2 - test_r2
if train_r2 < 0.7 and test_r2 < 0.7:
    status = "underfitting"
elif r2_gap > 0.05:
    status = "overfitting"
else:
    status = "good_fit"

metrics = {
    "train_r2": train_r2,
    "test_r2": test_r2,
    "train_mae": train_mae,
    "test_mae": test_mae,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "r2_gap": float(r2_gap),
}

# Optional training figure from the first visualization cell
training_visuals_fig_local = training_visuals_fig if "training_visuals_fig" in globals() else None

# Keep ID consistent with model/scaler exports in 01_surrogate_models
model_artifact_stem = build_model_artifact_stem(
    params["run_id"],
    params["learning_rate"],
    params["epochs"],
    final_val_r2,
)

architecture_summary = {
    "model_class": "TrussEdgeNNConv",
    "node_in_dim": len(schema.node_continuous_cols) + len(schema.node_mask_cols),
    "edge_in_dim": len(schema.edge_feature_cols),
    "global_in_dim": len(schema.global_feature_cols),
    "hidden_dim": params["hidden_dim"],
    "edge_count": schema.edge_count,
    "device": str(device),
    "dataset_sources": {
        "node": params["node_csv"],
        "edge": params["edge_csv"],
        "global": params["global_csv"],
    },
}

experiment_notes = (
    f"USE_PRETRAINED={params['use_pretrained']}; "
    f"lr={params['learning_rate']}; epochs={params['epochs']}; "
    f"batch_size={params['batch_size']}; hidden_dim={params['hidden_dim']}; "
    f"weight_decay={params['weight_decay']}"
)

# Save evaluation
saved_files = save_evaluation(
    model_prefix=model_artifact_stem,
    dataset_name=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    metrics=metrics,
    pred_residuals_fig=pred_residuals_fig,
    error_dist_fig=error_dist_fig,
    training_visuals_fig=training_visuals_fig_local,
    node_count=schema.node_count,
    edge_count=schema.edge_count,
    export_path=config.SM_DATA_PATH,
    status=status,
    run_id=params["run_id"],
    artifact_stem=model_artifact_stem,
    learning_rate=params["learning_rate"],
    epochs=params["epochs"],
    final_val_r2=final_val_r2,
    strict_dataset_label=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    source_dataset_path=str(config.GH_DATA_PATH / params["edge_csv"]),
    architecture_summary=architecture_summary,
    experiment_notes=experiment_notes,
    train_split_ratio=params["train_split_ratio"],
    random_seed=params["random_seed"],
    source_notebook="c21_surrogate_model_training.ipynb",
)

print(f"\n✅ Evaluation exported:")
for f in saved_files:
    print(f"  - {f}")


## Interpretation Guide

### What to look for:

**OVERFITTING** 🔴 (Train performs much better than Test):
- Train R² >> Test R² (gap > 0.05)
- Train residuals are much smaller than test residuals
- Test error histogram has a heavier right tail
- Predictions vs Actual scatter: train points closer to red line than test points

**UNDERFITTING** 🔴 (Both train and test perform poorly):
- Both R² scores are low (< 0.7)
- Both residuals show large systematic patterns
- Both predictions scatter far from the red diagonal line
- High MAE/RMSE on both train and test

**GOOD FIT** ✅ (Train and Test perform similarly):
- Train and Test R² are close (gap < 0.05)
- Both residuals are centered around 0 with similar spread
- Both scatter plots show points close to diagonal line
- Error distributions are similar and centered

### Remedies:

If **Overfitting**:
- **Gather more training data** (most effective long-term solution—model memorizes less with more diverse samples)
- Add dropout layers to the model
- Increase weight decay (L2 regularization)
- Use early stopping on validation loss
- Try reducing model hidden_dim (e.g., 128 → 64)

If **Underfitting**:
- Increase hidden_dim (e.g., 128 → 256)
- Add more GNN layers
- Train for more epochs
- Check if the model has enough capacity for the problem

If **Good Fit**: ✅ Deploy and use in downstream tasks!